

# Tools in Agentic AI Systems: A Principal-Level Technical Report

## Typed Infrastructure for External Action, Retrieval, and Closed-Loop Execution

---

## 1. Formal Definition and Architectural Role of Tools

### 1.1 Tools as Typed External Capability Interfaces

A **tool** in the agentic AI stack is a **typed, schema-described, externally-bound callable** that extends the computational closure of an LLM agent beyond text generation into observable state mutation, information retrieval, and environment interaction. Formally:

**Definition 1.1 (Tool).**
A tool $\mathcal{T}$ is a 7-tuple:

$$
\mathcal{T} = \langle \, \texttt{id}, \; \Sigma_{\text{in}}, \; \Sigma_{\text{out}}, \; \pi, \; \delta, \; \kappa, \; \tau_{\max} \, \rangle
$$

where:

| Symbol | Semantics |
|---|---|
| $\texttt{id}$ | Globally unique, versioned tool identifier (e.g., `flights/search@v2.3`) |
| $\Sigma_{\text{in}}$ | Input schema: a typed JSON Schema or Protobuf message descriptor specifying required and optional parameters, constraints, and validation rules |
| $\Sigma_{\text{out}}$ | Output schema: typed return envelope including structured result, error classes, pagination cursors, and provenance metadata |
| $\pi$ | Permission descriptor: mutation class (`read-only`, `state-mutating`, `irreversible`), required authorization scopes, and human-approval gates |
| $\delta$ | Description embedding: a high-signal natural-language description compiled for LLM tool-selection inference |
| $\kappa$ | Cost model: expected latency tier ($L_0 < 100\text{ms}$, $L_1 < 1\text{s}$, $L_2 < 10\text{s}$, $L_3$ async), token cost of injecting the schema into context, and monetary cost per invocation |
| $\tau_{\max}$ | Deadline: hard timeout with cancellation semantics |

**Definition 1.2 (Tool Registry).**
A tool registry $\mathcal{R}$ is a discoverable, versioned catalog:

$$
\mathcal{R} = \{ \mathcal{T}_1, \mathcal{T}_2, \ldots, \mathcal{T}_N \}
$$

exposed via **Model Context Protocol (MCP)** capability negotiation, supporting:

- **`tools/list`**: paginated enumeration with cursor-based traversal
- **`tools/call`**: schema-validated invocation with deadline propagation
- **`notifications/tools/list_changed`**: push-based invalidation on registry mutation

The registry is the **single source of truth** for agent-accessible capabilities. No tool is callable unless registered, schema-validated, and authorization-scoped.

### 1.2 Why Tools Are Architecturally Non-Negotiable

An LLM without tools is a **closed-world reasoner** operating over a frozen parametric snapshot $\theta$ trained at time $t_{\text{cutoff}}$. The information-theoretic limitation is precise:

$$
I_{\text{agent}}(q, t) = \underbrace{I_{\theta}(q)}_{\text{parametric knowledge}} + \underbrace{I_{\mathcal{R}}(q, t)}_{\text{tool-retrieved, real-time}}
$$

For any query $q$ with temporal dependency $t > t_{\text{cutoff}}$ or requiring external state observation, $I_{\theta}(q) \approx 0$ and the agent **must** delegate to a tool. Tools thus transform the agent from a **static text generator** into a **dynamic actuator** capable of:

1. **Observation**: querying live databases, APIs, sensors, browser state, file systems
2. **Mutation**: writing records, sending messages, deploying code, modifying infrastructure
3. **Verification**: running tests, validating outputs against ground truth, inspecting execution traces

---

## 2. Protocol Stack: Typed Boundaries for Tool Exposure

### 2.1 Three-Tier Protocol Architecture

Tools are **not** exposed as ad hoc function names injected into a prompt string. They are served through a **typed protocol stack** with explicit schemas, error classes, versioned contracts, and deadline propagation:

```
┌─────────────────────────────────────────────────────┐
│  Layer 3: JSON-RPC 2.0 — User / Application Boundary│
│  • Human-facing API surface                          │
│  • Request/response with structured error codes      │
│  • Pagination, capability discovery                  │
├─────────────────────────────────────────────────────┤
│  Layer 2: MCP (Model Context Protocol) — Tool Plane  │
│  • tools/list, tools/call, resources/read            │
│  • Schema-described inputs/outputs                   │
│  • Change notifications, capability negotiation      │
│  • Local (stdio) and remote (SSE/HTTP) transports    │
├─────────────────────────────────────────────────────┤
│  Layer 1: gRPC / Protobuf — Internal Execution Plane │
│  • Low-latency service-to-service calls              │
│  • Strongly typed message contracts                  │
│  • Streaming, deadline propagation, load balancing   │
│  • mTLS, per-RPC authorization                       │
└─────────────────────────────────────────────────────┘
```

**Design Rationale:**

| Layer | Why This Protocol | Trade-off |
|---|---|---|
| **JSON-RPC** | Human-readable, universal client support, natural fit for LLM-generated structured output | Higher serialization overhead vs. binary protocols |
| **MCP** | Purpose-built for LLM tool discovery; supports `tools`, `resources`, `prompts` as first-class primitives with schema negotiation | Emerging standard; requires MCP server implementation per tool provider |
| **gRPC** | Sub-millisecond internal dispatch, strongly typed Protobuf contracts, native streaming, built-in deadline/cancellation propagation | Requires code generation; not human-readable on wire |

### 2.2 Tool Contract Specification (Protobuf + MCP)

```protobuf
// Internal gRPC contract for tool execution
syntax = "proto3";
package agentic.tools.v2;

message ToolInvocationRequest {
  string tool_id = 1;                    // e.g., "flights/search@v2.3"
  string idempotency_key = 2;           // Client-generated UUID for retry safety
  google.protobuf.Struct arguments = 3; // Schema-validated input
  string caller_scope = 4;              // Authorization scope of invoking agent
  google.protobuf.Duration deadline = 5; // Hard timeout
  map<string, string> trace_context = 6; // OpenTelemetry W3C trace propagation
}

message ToolInvocationResponse {
  oneof result {
    google.protobuf.Struct content = 1;  // Structured output
    ToolError error = 2;                 // Typed error
  }
  ProvenanceMetadata provenance = 3;     // Source, timestamp, authority
  PaginationCursor next_cursor = 4;     // For paginated results
  CostReport cost = 5;                  // Tokens consumed, latency, monetary cost
}

message ToolError {
  enum ErrorClass {
    INVALID_ARGUMENTS = 0;
    PERMISSION_DENIED = 1;
    RATE_LIMITED = 2;
    UPSTREAM_FAILURE = 3;
    TIMEOUT = 4;
    HUMAN_APPROVAL_REQUIRED = 5;
  }
  ErrorClass class = 1;
  string message = 2;
  bool retryable = 3;
  google.protobuf.Duration retry_after = 4;
}
```

```json
// MCP tool declaration (served via tools/list)
{
  "name": "search_flights",
  "version": "2.3.0",
  "description": "Search real-time flight availability between airports. Returns ranked options by price, duration, and stops. Use when the user requests flight booking, comparison, or availability checking. Do NOT use for hotel or ground transport queries.",
  "inputSchema": {
    "type": "object",
    "properties": {
      "origin": { "type": "string", "pattern": "^[A-Z]{3}$", "description": "IATA airport code of departure" },
      "destination": { "type": "string", "pattern": "^[A-Z]{3}$", "description": "IATA airport code of arrival" },
      "departure_date": { "type": "string", "format": "date", "description": "ISO 8601 date" },
      "passengers": { "type": "integer", "minimum": 1, "maximum": 9 },
      "cabin_class": { "type": "string", "enum": ["economy", "business", "first"], "default": "economy" }
    },
    "required": ["origin", "destination", "departure_date"]
  },
  "outputSchema": { "$ref": "#/definitions/FlightSearchResult" },
  "mutationClass": "read-only",
  "latencyTier": "L1",
  "costPerCall": { "usd": 0.002 },
  "requiresApproval": false
}
```

**Critical Observation**: The `description` field is the **highest-leverage context engineering artifact** for tool selection. It functions as a **compiled directive** to the LLM's tool-routing inference. It must contain:

1. **Positive specification**: what the tool does, precisely
2. **Negative specification**: what the tool does NOT do (disambiguation from sibling tools)
3. **Trigger conditions**: when to invoke (intent patterns)
4. **Input constraints**: type, format, valid ranges — redundant with schema but required for LLM grounding

---

## 3. Context Engineering for Tool Affordances

### 3.1 The Tool Context Budget Problem

Injecting all $N$ tool schemas into the LLM context window is **infeasible** at scale. Each tool schema consumes $c_i$ tokens. The total tool context cost is:

$$
C_{\text{tools}} = \sum_{i=1}^{N} c_i
$$

For an enterprise registry with $N = 500$ tools and average schema size $\bar{c} = 200$ tokens, $C_{\text{tools}} = 100{,}000$ tokens — exceeding most context windows before any task context, memory, or retrieval payload is loaded.

**Theorem 3.1 (Tool Budget Constraint).**
Given a context window of $W$ tokens and allocations for system policy ($B_{\text{sys}}$), task state ($B_{\text{task}}$), retrieval evidence ($B_{\text{ret}}$), memory ($B_{\text{mem}}$), and reasoning reserve ($B_{\text{res}}$), the tool budget is:

$$
B_{\text{tools}} = W - B_{\text{sys}} - B_{\text{task}} - B_{\text{ret}} - B_{\text{mem}} - B_{\text{res}}
$$

The set of tools injected into context, $\mathcal{T}_{\text{active}} \subseteq \mathcal{R}$, must satisfy:

$$
\sum_{\mathcal{T}_i \in \mathcal{T}_{\text{active}}} c_i \leq B_{\text{tools}}
$$

This is a **constrained selection optimization** problem.

### 3.2 Lazy Tool Loading via Relevance-Gated Selection

**SOTA Approach: Two-Phase Tool Resolution**

Tools are **never** bulk-loaded. Instead, a **tool router** selects the minimal relevant subset per turn.

**Phase 1 — Intent-Based Pre-Filter (Sub-Linear Scan)**

Given user query $q$, compute a task intent embedding $\mathbf{e}_q = \text{Embed}(q)$ and retrieve candidate tools by approximate nearest-neighbor search over pre-computed tool description embeddings:

$$
\mathcal{T}_{\text{candidates}} = \text{ANN}\!\left(\mathbf{e}_q, \; \{ \mathbf{e}_{\delta_i} \}_{i=1}^{N}, \; k \right)
$$

where $k$ is a retrieval budget (typically $k \in [5, 15]$).

**Phase 2 — Schema-Aware Re-Ranking**

Re-rank candidates using a scoring function that incorporates semantic relevance, historical invocation success, cost, and mutation safety:

$$
\text{score}(\mathcal{T}_i \mid q, \text{ctx}) = \underbrace{\alpha \cdot \text{sim}(\mathbf{e}_q, \mathbf{e}_{\delta_i})}_{\text{semantic relevance}} + \underbrace{\beta \cdot P(\text{success} \mid \mathcal{T}_i, q)}_{\text{historical success rate}} - \underbrace{\gamma \cdot \kappa_i}_{\text{cost penalty}} - \underbrace{\lambda \cdot \mathbb{1}[\pi_i = \text{mutating}]}_{\text{mutation risk}}
$$

Select $\mathcal{T}_{\text{active}}$ by solving:

$$
\mathcal{T}_{\text{active}}^{*} = \arg\max_{\mathcal{S} \subseteq \mathcal{T}_{\text{candidates}}} \sum_{\mathcal{T}_i \in \mathcal{S}} \text{score}(\mathcal{T}_i \mid q, \text{ctx}) \quad \text{s.t.} \quad \sum_{\mathcal{T}_i \in \mathcal{S}} c_i \leq B_{\text{tools}}
$$

This is a variant of the **0-1 knapsack problem** (NP-hard in general, but tractable for small $k$ via greedy approximation with guaranteed $1 - 1/e$ optimality ratio).

```
Algorithm 1: Lazy Tool Loading with Budget-Constrained Selection
─────────────────────────────────────────────────────────────────
Input: query q, registry R, token budget B_tools
Output: T_active (set of tool schemas to inject)

1.  e_q ← Embed(q)
2.  T_candidates ← ANN(e_q, {e_δ_i}_{i=1}^N, k=15)
3.  for each T_i in T_candidates:
4.      s_i ← α·sim(e_q, e_δ_i) + β·P_success(T_i, q) − γ·κ_i − λ·𝟙[mutating]
5.  Sort T_candidates by s_i descending
6.  T_active ← ∅; budget_used ← 0
7.  for each T_i in sorted T_candidates:
8.      if budget_used + c_i ≤ B_tools:
9.          T_active ← T_active ∪ {T_i}
10.         budget_used ← budget_used + c_i
11. return T_active
```

**Complexity**: $O(N)$ for embedding pre-computation (offline), $O(\log N)$ for ANN retrieval, $O(k \log k)$ for re-ranking. Total per-turn cost: **sub-millisecond**.

### 3.3 Tool Description as Compiled Directive

The tool description $\delta_i$ is the **primary signal** the LLM uses for tool selection. It must be engineered as a **compiled directive**, not prose documentation.

**SOTA Description Template:**

```
[WHAT] {Precise capability in one sentence.}
[WHEN] {Exact trigger conditions — user intents that SHOULD activate this tool.}
[WHEN NOT] {Explicit negative triggers — intents that SHOULD NOT activate this tool, with redirect to correct tool.}
[INPUTS] {Critical parameter semantics beyond what schema types convey.}
[OUTPUTS] {What the return payload represents; how to interpret absence/emptiness.}
[SIDE EFFECTS] {State mutations, if any. "None" for read-only tools.}
[CHAIN HINTS] {Which tools commonly precede or follow this one in multi-step workflows.}
```

**Example (SOTA-compiled):**

```
[WHAT] Search real-time flight availability between two airports on a specific date.
[WHEN] User asks to find, compare, or check availability of flights.
[WHEN NOT] Do NOT use for hotel search (→ search_hotels), car rental (→ search_cars), or flight booking/purchase (→ book_flight). Do NOT use if user already has flight details and wants to modify (→ modify_booking).
[INPUTS] origin/destination must be valid 3-letter IATA codes. If user provides city name, resolve to IATA code using resolve_airport_code FIRST.
[OUTPUTS] Returns ranked list of flight options with price, duration, stops, carrier. Empty list means no availability — inform user and suggest adjacent dates.
[SIDE EFFECTS] None (read-only).
[CHAIN HINTS] Often preceded by resolve_airport_code. Often followed by book_flight or search_hotels.
```

**Theorem 3.2 (Description Disambiguation Necessity).**
Given $n$ tools with overlapping semantic domains, the probability of tool misselection increases polynomially with domain overlap $\omega_{ij}$ between tool pairs $(i, j)$:

$$
P(\text{misselect}) \propto \sum_{i < j} \omega_{ij}^2
$$

Negative specifications (`[WHEN NOT]`) reduce $\omega_{ij}$ by providing **contrastive signal** that sharpens the decision boundary in the LLM's inference space. Empirically, adding negative specifications reduces tool misselection by **35–60%** on benchmarks like ToolBench and API-Bank.

---

## 4. The Evolution: From Prompt Hacking to Typed Function Calling

### 4.1 Generation Taxonomy

The evolution of tool invocation in LLM systems follows a strict capability ladder:

| Generation | Mechanism | Failure Mode | Reliability |
|---|---|---|---|
| **Gen-0: Regex Extraction** | Prompt instructs LLM to emit text matching a regex pattern (e.g., `ACTION: search("tokyo")`) | Fragile parsing; format drift; no type safety | ~60% parse success |
| **Gen-1: ReAct-Style Prompting** | Chain-of-thought + action/observation loop with text-based tool calls | Better reasoning but still text-parsed; hallucinated tool names/args | ~75% |
| **Gen-2: Native Function Calling** | LLM outputs structured JSON with function name and typed arguments as a **dedicated output mode** distinct from text generation | Mismatched schemas; argument type errors | ~90% |
| **Gen-3: Typed Protocol Binding (SOTA)** | Function calling + MCP discovery + schema validation + idempotency + approval gates + provenance | Requires infrastructure investment | ~98%+ with verification |

### 4.2 Native Function Calling: Formal Model

In Gen-2+ systems, the LLM operates as a **policy function** $\pi_\theta$ that, given context $\mathbf{c}$, produces either a **text response** $r$ or a **tool call** $f$:

$$
\pi_\theta(\mathbf{c}) \to \begin{cases} r \in \Sigma^* & \text{(text response)} \\ f = (\texttt{tool\_id}, \; \texttt{args} \in \Sigma_{\text{in}}) & \text{(tool invocation)} \end{cases}
$$

The training objective for function calling fine-tunes the model to emit valid JSON conforming to the declared schemas with high probability:

$$
\mathcal{L}_{\text{fc}} = -\mathbb{E}_{(q, \mathcal{T}_{\text{active}})}\left[\log P_\theta\!\left(f^* \mid q, \; \{\delta_i, \Sigma_{\text{in},i}\}_{i \in \mathcal{T}_{\text{active}}}\right)\right]
$$

where $f^*$ is the ground-truth tool call.

**Critical distinction from text generation**: The output mode switch is a **structured decode** — the model's logits are constrained (via guided/constrained decoding or fine-tuned output heads) to produce JSON tokens that conform to the declared input schema. This is fundamentally different from hoping the model "guesses" the right format.

### 4.3 Parallel and Chained Tool Calling

Modern SOTA models support **multiple tool calls per turn**, enabling:

**Parallel Invocation** (independent tools, no data dependency):

$$
\{f_1, f_2, \ldots, f_m\} = \pi_\theta(\mathbf{c}) \quad \text{where} \quad \forall i \neq j: \; \text{args}(f_j) \not\!\perp\!\!\!\perp \text{output}(f_i)
$$

All $m$ calls execute concurrently. Results are aggregated and re-injected into context.

**Sequential Chaining** (data-dependent tools):

$$
f_1 = \pi_\theta(\mathbf{c}_0), \quad \mathbf{c}_1 = \mathbf{c}_0 \oplus \text{result}(f_1), \quad f_2 = \pi_\theta(\mathbf{c}_1), \quad \ldots
$$

Each step's output enriches the context for the next decision. The chain length $L$ is **bounded**:

$$
L \leq L_{\max} \quad \text{(recursion depth bound to prevent runaway execution)}
$$

---

## 5. Tool Invocation as a Bounded Control Loop

### 5.1 The Complete Tool Execution Cycle

Tool invocation is **never** a fire-and-forget operation. It is embedded within a **closed-loop control system** with verification, error handling, and repair:

```
Algorithm 2: Tool Invocation Control Loop
──────────────────────────────────────────
Input: agent context c, tool registry R, max_iterations L_max
Output: final response r or failure state F

1.  iteration ← 0
2.  while iteration < L_max:
3.      // PLAN: Determine if a tool call is needed
4.      action ← π_θ(c)
5.      if action is TextResponse r:
6.          return r   // Terminal: agent has enough information
7.      
8.      // VALIDATE: Schema-check the proposed tool call
9.      f = (tool_id, args) ← action
10.     if tool_id ∉ R:
11.         c ← c ⊕ Error("Tool not found: {tool_id}. Available: {R.list()}")
12.         iteration += 1; continue
13.     if not validate(args, Σ_in[tool_id]):
14.         c ← c ⊕ Error("Invalid arguments: {validation_errors}")
15.         iteration += 1; continue
16.     
17.     // AUTHORIZE: Check permissions and approval gates
18.     if T[tool_id].π requires_approval and not human_approved(f):
19.         c ← c ⊕ PendingApproval(f)
20.         yield HumanApprovalRequest(f)  // Suspend execution
21.         // Resume on approval or rejection
22.     
23.     // EXECUTE: Invoke with deadline, idempotency, and circuit breaker
24.     idem_key ← generate_idempotency_key(f, c.session_id)
25.     result ← execute_with_retry(
26.         tool_id, args, idem_key,
27.         deadline=T[tool_id].τ_max,
28.         retry_budget=3,
29.         backoff=exponential_with_jitter
30.     )
31.     
32.     // VERIFY: Check result validity
33.     if result is ToolError:
34.         if result.retryable and retry_budget > 0:
35.             retry_budget -= 1; continue
36.         c ← c ⊕ ErrorContext(result)
37.         iteration += 1; continue
38.     
39.     // OBSERVE: Inject result with provenance into context
40.     c ← c ⊕ ToolResult(
41.         tool_id, result.content, result.provenance,
42.         timestamp=now(), latency=result.cost.latency
43.     )
44.     
45.     // CRITIQUE: Let the agent evaluate whether the result is sufficient
46.     iteration += 1
47.     // Loop continues — agent will decide to call another tool or respond
48.
49.  // Exceeded iteration bound
50.  return FailureState("Max iterations reached", context=c)
```

### 5.2 Formal Properties of the Control Loop

**Property 1: Termination Guarantee.**
The loop terminates in at most $L_{\max}$ iterations. This prevents unbounded tool-call chains that consume tokens, cost, and latency without convergence.

$$
\forall \text{execution}: \; |\text{iterations}| \leq L_{\max}
$$

**Property 2: Idempotency.**
Every state-mutating tool call carries an idempotency key $k = H(\texttt{tool\_id} \| \texttt{args} \| \texttt{session\_id})$. Retries with the same key produce identical effects:

$$
\text{execute}(f, k) = \text{execute}(f, k) \quad \forall \text{retries}
$$

**Property 3: Monotonic Context Enrichment.**
Each iteration either adds a tool result (information gain) or an error context (diagnostic gain) to $\mathbf{c}$. The information content of context is non-decreasing:

$$
H(\mathbf{c}_{t+1} \mid q) \geq H(\mathbf{c}_t \mid q)
$$

**Property 4: Human Interruptibility.**
Any state-mutating path with `requires_approval = true` **suspends** execution and yields control to a human operator. The agent cannot bypass this gate.

---

## 6. Tool Chaining: Multi-Step Workflow Orchestration

### 6.1 Dependency Graph Formalization

A complex user request decomposes into a **directed acyclic graph (DAG)** of tool invocations:

**Definition 6.1 (Tool Execution DAG).**
Given a task $T$, the execution plan is a DAG $G = (V, E)$ where:
- $V = \{v_1, \ldots, v_n\}$ are tool invocation nodes
- $E \subseteq V \times V$ are data-dependency edges: $(v_i, v_j) \in E$ iff $v_j$ requires output from $v_i$

The **critical path latency** of the plan is:

$$
\Lambda(G) = \max_{\text{path } p \in G} \sum_{v \in p} \tau(v)
$$

where $\tau(v)$ is the expected latency of tool $v$.

**Example: "Plan a weekend trip to San Francisco"**

```
                    ┌──────────────────┐
                    │  resolve_city    │
                    │  "San Francisco" │
                    │  → IATA: SFO     │
                    └────────┬─────────┘
                             │
              ┌──────────────┼──────────────┐
              │              │              │
              ▼              ▼              ▼
     ┌────────────┐  ┌────────────┐  ┌─────────────┐
     │search_flights│ │search_hotels│ │get_local    │
     │origin=USER  │  │city=SFO    │  │  _events    │
     │dest=SFO     │  │checkin=FRI │  │city=SFO     │
     │date=FRI     │  │checkout=SUN│  │dates=FRI-SUN│
     └──────┬─────┘  └──────┬─────┘  └──────┬──────┘
            │               │               │
            └───────────────┼───────────────┘
                            │
                            ▼
                   ┌────────────────┐
                   │ synthesize_plan │
                   │ (LLM reasoning) │
                   └────────────────┘
```

**Parallelism Extraction**: `search_flights`, `search_hotels`, and `get_local_events` are **independent** given the resolved city. They execute **concurrently**, reducing wall-clock latency from $\sum \tau_i$ to $\max(\tau_i)$.

### 6.2 Automatic Plan Decomposition

```
Algorithm 3: Task Decomposition and Tool-Chain Planning
───────────────────────────────────────────────────────
Input: user query q, tool registry R, agent policy π_θ
Output: execution DAG G = (V, E)

1.  // DECOMPOSE: Break query into atomic sub-tasks
2.  subtasks ← π_θ(
3.      system="Decompose the user request into atomic sub-tasks. "
4.             "Each sub-task must map to exactly one tool or a synthesis step. "
5.             "Specify data dependencies between sub-tasks.",
6.      user=q,
7.      tools=T_active   // Lazy-loaded relevant subset
8.  )
9.  // Parse structured decomposition output
10. V ← {v_i : subtask_i ∈ subtasks}
11. E ← {(v_i, v_j) : v_j.depends_on contains v_i}
12.
13. // VALIDATE: Check DAG properties
14. assert is_acyclic(V, E)          // No circular dependencies
15. assert ∀ v ∈ V: v.tool_id ∈ R   // All tools exist
16. assert |V| ≤ V_max              // Bounded plan complexity
17.
18. // OPTIMIZE: Identify parallelizable groups
19. levels ← topological_sort_levels(V, E)
20. for each level L in levels:
21.     // Tools within the same level have no inter-dependencies
22.     // → Execute concurrently
23.     mark_parallel(L)
24.
25. return G = (V, E, levels)
```

### 6.3 Execution Engine with Concurrency Control

```
Algorithm 4: DAG Executor with Parallel Tool Dispatch
─────────────────────────────────────────────────────
Input: DAG G = (V, E, levels), context c
Output: aggregated results R_all, updated context c'

1.  R_all ← {}
2.  for each level L in topological order:
3.      // Dispatch all tools in this level concurrently
4.      futures ← {}
5.      for each v in L:
6.          // Resolve arguments: substitute outputs from predecessor nodes
7.          resolved_args ← resolve_references(v.args, R_all)
8.          fut ← async_execute(v.tool_id, resolved_args,
9.                              deadline=v.τ_max, idem_key=v.idem_key)
10.         futures[v.id] ← fut
11.     
12.     // Await all with deadline; handle partial failures
13.     results ← await_all_with_timeout(futures, deadline=max(v.τ_max for v in L))
14.     for each (v_id, result) in results:
15.         if result is Error:
16.             // Compensating action: retry, fallback, or skip with degraded quality
17.             result ← handle_tool_failure(v_id, result, compensation_policy)
18.         R_all[v_id] ← result
19.     
20.  // Inject all results into context with provenance
21.  c' ← c ⊕ {ToolResult(v_id, R_all[v_id]) for v_id in V}
22.  return R_all, c'
```

**Key Properties:**

- **Maximum parallelism**: Independent tools in the same topological level execute concurrently
- **Minimum latency**: Critical path latency $\Lambda(G)$ is achieved, not sequential sum
- **Fault isolation**: Failure of one tool in a level does not block siblings; compensating actions are applied per-node

---

## 7. Reliability Engineering for Tool Invocation

### 7.1 Retry Budget with Exponential Backoff and Jitter

For transient failures (network timeouts, rate limits), apply bounded retries:

$$
t_{\text{wait}}(n) = \min\!\left(\text{base} \cdot 2^n + \text{Uniform}(0, \text{jitter}), \; t_{\max}\right)
$$

where $n$ is the retry attempt number. The **retry budget** $R_b$ is the maximum number of retries per tool call. Total maximum latency per call:

$$
\tau_{\text{total}} \leq \tau_{\text{call}} + \sum_{n=1}^{R_b} t_{\text{wait}}(n) + \tau_{\text{call}}
$$

### 7.2 Circuit Breaker Pattern

For persistent failures (upstream service down), apply a **circuit breaker** to prevent cascading failures:

$$
\text{state}(\mathcal{T}_i) = \begin{cases}
\texttt{CLOSED} & \text{if } \text{fail\_rate}(\mathcal{T}_i, w) < \theta_{\text{open}} \\
\texttt{OPEN} & \text{if } \text{fail\_rate}(\mathcal{T}_i, w) \geq \theta_{\text{open}} \\
\texttt{HALF\_OPEN} & \text{after cooldown } \Delta t_{\text{cool}}
\end{cases}
$$

where $w$ is the observation window and $\theta_{\text{open}}$ is the failure-rate threshold (e.g., 50% over 60 seconds). In `OPEN` state, all calls to $\mathcal{T}_i$ are **immediately rejected** with a `CIRCUIT_OPEN` error, allowing the agent to select fallback tools or degrade gracefully.

### 7.3 Hallucination Control in Tool Usage

Tool-related hallucinations fall into three categories:

| Hallucination Type | Definition | Mitigation |
|---|---|---|
| **Phantom Tool** | LLM invokes a tool not in $\mathcal{R}$ | Schema-constrained decoding; validate `tool_id ∈ R` before execution |
| **Argument Fabrication** | LLM fills required arguments with plausible but invented values | JSON Schema validation; reject calls with `INVALID_ARGUMENTS`; re-prompt with explicit error |
| **Result Confabulation** | LLM ignores actual tool output and generates a plausible-sounding answer from parametric knowledge | Architectural enforcement: tool results injected as **system-role messages** with provenance tags; verification step compares response claims against tool output |

**Formal Mitigation: Grounded Response Verification**

After tool execution and response generation, apply a **grounding check**:

$$
\text{grounded}(r, \mathcal{O}) = \frac{|\text{claims}(r) \cap \text{facts}(\mathcal{O})|}{|\text{claims}(r)|}
$$

where $r$ is the agent's response and $\mathcal{O}$ is the set of tool outputs. If $\text{grounded}(r, \mathcal{O}) < \theta_{\text{ground}}$ (e.g., 0.9), the response is **rejected** and regenerated with a strengthened grounding instruction.

```
Algorithm 5: Grounded Response Verification
────────────────────────────────────────────
Input: response r, tool outputs O, threshold θ_ground
Output: verified response r' or rejection

1.  claims ← extract_factual_claims(r)  // NLI-based claim extraction
2.  for each claim c in claims:
3.      supported ← FALSE
4.      for each output o in O:
5.          if entails(o, c):   // NLI entailment check
6.              supported ← TRUE; break
7.      if not supported:
8.          mark_ungrounded(c)
9.  
10. grounding_score ← |supported_claims| / |claims|
11. if grounding_score < θ_ground:
12.     // Regenerate with explicit grounding instruction
13.     r' ← π_θ(c ⊕ "Your response contained ungrounded claims. "
14.                    "Use ONLY the tool outputs below. Do not add information "
15.                    "not present in these results." ⊕ O)
16.     return verify(r', O, θ_ground)  // Recursive, bounded
17. return r
```

---

## 8. Authorization, Least Privilege, and Human Governance

### 8.1 Scoped Authorization Model

Tools are bound to **caller-scoped authorization**, not broad agent credentials:

$$
\text{authorized}(f, \text{agent}, \text{user}) = \text{scope}(\text{user}) \cap \text{requires}(\mathcal{T}_{f.\text{id}}) \neq \emptyset
$$

The agent **inherits** the invoking user's permission set. It cannot escalate privileges.

### 8.2 Mutation Classification and Approval Gates

Every tool is classified on a **mutation severity spectrum**:

| Class | Example | Approval Policy |
|---|---|---|
| `read-only` | `search_flights`, `get_weather` | Auto-approved |
| `state-mutating-reversible` | `add_to_cart`, `create_draft` | Auto-approved with undo window |
| `state-mutating-irreversible` | `book_flight`, `send_payment` | **Human approval required** |
| `destructive` | `delete_account`, `cancel_subscription` | **Explicit double-confirmation** |

```
Algorithm 6: Mutation-Aware Tool Gating
───────────────────────────────────────
Input: tool call f, mutation_class π
Output: execution permission or suspension

1.  switch π:
2.      case READ_ONLY:
3.          return PERMIT
4.      case MUTATING_REVERSIBLE:
5.          log_with_undo_window(f, undo_ttl=300s)
6.          return PERMIT
7.      case MUTATING_IRREVERSIBLE:
8.          summary ← generate_human_summary(f)
9.          approval ← request_human_approval(summary, timeout=600s)
10.         if approval == APPROVED:
11.             return PERMIT
12.         elif approval == TIMEOUT:
13.             return DENY(reason="Approval timeout")
14.         else:
15.             return DENY(reason=approval.reason)
16.     case DESTRUCTIVE:
17.         require_double_confirmation(f)
18.         return conditional_permit()
```

---

## 9. Observability and Audit Infrastructure

### 9.1 Trace Structure for Tool Invocations

Every tool call produces a **structured trace span** compliant with OpenTelemetry:

$$
\text{Span}(\mathcal{T}_i) = \langle \texttt{trace\_id}, \; \texttt{span\_id}, \; \texttt{parent\_span}, \; \texttt{tool\_id}, \; \texttt{args\_hash}, \; \texttt{start}, \; \texttt{end}, \; \texttt{status}, \; \texttt{result\_hash}, \; \texttt{cost} \rangle
$$

**Metrics collected per tool:**

| Metric | Aggregation | Alert Threshold |
|---|---|---|
| `tool.invocation.count` | Counter per tool_id, per agent, per user | Anomaly detection on rate |
| `tool.latency.p50/p95/p99` | Histogram | $p_{99} > 2 \times \tau_{\max}$ |
| `tool.error.rate` | Rate per window | $> \theta_{\text{open}}$ triggers circuit breaker |
| `tool.misselection.rate` | Counter (detected via critique step) | $> 10\%$ triggers description rewrite |
| `tool.token.cost` | Sum of schema tokens injected | Budget overshoot alert |
| `tool.result.grounding` | Grounding score distribution | $\text{mean} < 0.85$ triggers pipeline review |

### 9.2 Audit Trail for Compliance

All tool invocations are persisted in an **append-only audit log**:

```
{
  "timestamp": "2025-01-15T14:23:07.831Z",
  "trace_id": "abc123...",
  "agent_id": "travel-planner-v3",
  "user_id": "user_9182",
  "tool_id": "flights/search@v2.3",
  "arguments": { "origin": "JFK", "destination": "SFO", "date": "2025-01-24" },
  "arguments_hash": "sha256:e3b0c442...",
  "result_summary": "5 flights found, cheapest $189",
  "result_hash": "sha256:d7a8fbb3...",
  "latency_ms": 342,
  "cost_usd": 0.002,
  "mutation_class": "read-only",
  "approval_required": false,
  "idempotency_key": "idem_8f3a...",
  "status": "SUCCESS"
}
```

---

## 10. Cost Optimization and Token Efficiency

### 10.1 Schema Compression

Tool schemas injected into context are **compressed** to minimize token consumption while preserving selection signal:

**Technique: Progressive Disclosure**

```
Level 0 (Discovery):     tool_name + one-line description       (~20 tokens)
Level 1 (Selection):     + required params + trigger conditions  (~80 tokens)
Level 2 (Invocation):    Full schema with all params, types,     (~200 tokens)
                          constraints, examples
```

At **Phase 1** (intent filtering), inject Level 0 descriptions for all candidates. At **Phase 2** (after selection), inject Level 2 only for the selected tool. Token savings:

$$
\Delta C = \sum_{i \in \mathcal{T}_{\text{candidates}} \setminus \mathcal{T}_{\text{active}}} (c_i^{L2} - c_i^{L0})
$$

For 15 candidates with 1–3 selected: savings of **~2,500 tokens per turn**.

### 10.2 Result Caching

Tool results are cached with content-addressed keys and TTL:

$$
\text{key} = H(\texttt{tool\_id} \| \texttt{canonical\_args})
$$

$$
\text{cache\_hit}(f) = \begin{cases}
\text{cached\_result} & \text{if } \text{age}(\text{cached\_result}) < \text{TTL}(\mathcal{T}_{f.\text{id}}) \\
\bot & \text{otherwise}
\end{cases}
$$

TTL is set per tool based on data volatility:

| Tool Category | TTL | Rationale |
|---|---|---|
| Static reference data | 24h | Airport codes, currency symbols |
| Semi-dynamic | 15min | Hotel availability, event listings |
| Real-time | 0 (no cache) | Stock prices, flight status |

---

## 11. End-to-End Prefill Compilation with Tool Context

The **prefill compiler** assembles the complete agent context as a deterministic artifact:

```
Algorithm 7: Prefill Compilation with Tool Affordances
──────────────────────────────────────────────────────
Input: query q, session s, memory M, retrieval R, policy P
Output: compiled context c (token-budgeted)

1.  // Allocate token budgets
2.  W ← model.context_window              // e.g., 128K
3.  B_sys ← 500                            // System policy
4.  B_res ← W * 0.25                       // Reasoning reserve (25%)
5.  B_remaining ← W - B_sys - B_res
6.
7.  // Phase 1: System policy (always included, highest priority)
8.  c ← compile_system_policy(P)           // Role, constraints, output format
9.
10. // Phase 2: Tool affordances (lazy-loaded)
11. T_active ← lazy_tool_load(q, R, B_tools=min(B_remaining*0.15, 3000))
12. c ← c ⊕ compile_tool_schemas(T_active, level=2)
13. B_remaining -= token_count(T_active)
14.
15. // Phase 3: Memory summaries
16. mem_summary ← compile_memory(M, budget=min(B_remaining*0.10, 1000))
17. c ← c ⊕ mem_summary
18. B_remaining -= token_count(mem_summary)
19.
20. // Phase 4: Retrieved evidence (provenance-tagged)
21. evidence ← retrieve_and_rank(q, budget=min(B_remaining*0.30, 4000))
22. c ← c ⊕ compile_evidence(evidence)
23. B_remaining -= token_count(evidence)
24.
25. // Phase 5: Conversation history (compressed, most recent first)
26. history ← compress_history(s.messages, budget=B_remaining)
27. c ← c ⊕ history
28.
29. // Phase 6: Current query
30. c ← c ⊕ format_query(q)
31.
32. assert token_count(c) ≤ W - B_res    // Hard invariant
33. return c
```

**Token Budget Allocation (128K window):**

```
┌──────────────────────────────────────────────────┐
│ System Policy          │    500 tokens  (0.4%)   │
│ Tool Schemas (active)  │  3,000 tokens  (2.3%)   │
│ Memory Summaries       │  1,000 tokens  (0.8%)   │
│ Retrieved Evidence     │  4,000 tokens  (3.1%)   │
│ Conversation History   │ 87,500 tokens  (68.4%)  │
│ Current Query          │  ~500 tokens   (0.4%)   │
│ ─── Reasoning Reserve ─│ 32,000 tokens  (25.0%)  │
└──────────────────────────────────────────────────┘
```

---

## 12. Production Deployment Architecture

```
                           ┌────────────────────┐
                           │   Client / UI      │
                           │  (JSON-RPC 2.0)    │
                           └────────┬───────────┘
                                    │
                           ┌────────▼───────────┐
                           │  API Gateway        │
                           │  Rate limit, AuthN  │
                           │  Trace injection    │
                           └────────┬───────────┘
                                    │
                    ┌───────────────▼───────────────┐
                    │   Agent Orchestrator           │
                    │   (Control loop: Algo 2)       │
                    │   Plan → Act → Verify → Commit │
                    └──┬────────┬────────┬──────────┘
                       │        │        │
              ┌────────▼──┐  ┌──▼─────┐  ┌▼──────────────┐
              │  Prefill   │  │ Tool   │  │  Memory       │
              │  Compiler  │  │ Router │  │  Manager      │
              │  (Algo 7)  │  │(Algo 1)│  │  (write/read) │
              └────────────┘  └──┬─────┘  └───────────────┘
                                 │
                    ┌────────────▼────────────┐
                    │   MCP Tool Gateway       │
                    │   Schema validation      │
                    │   Auth scoping           │
                    │   Circuit breaker        │
                    │   Idempotency cache      │
                    └──┬──────┬──────┬────────┘
                       │      │      │
              ┌────────▼┐  ┌──▼───┐  ┌▼──────────┐
              │ Tool     │  │ Tool │  │ Tool      │
              │ Server A │  │ Srv B│  │ Server C  │
              │ (gRPC)   │  │(gRPC)│  │ (MCP/SSE) │
              └──────────┘  └──────┘  └───────────┘
```

### 12.1 Operational Constraints

| Constraint | Target | Mechanism |
|---|---|---|
| **Latency** | Tool selection < 50ms; total tool chain < 10s | ANN index, pre-computed embeddings, parallel dispatch |
| **Cost** | < $0.05 per agent turn (tool overhead) | Schema compression, caching, lazy loading |
| **Reliability** | 99.9% successful tool resolution | Circuit breakers, fallback tools, retry budgets |
| **Token Efficiency** | Tool context < 5% of window | Progressive disclosure, budget-constrained selection |
| **Safety** | Zero unauthorized mutations | Scoped auth, approval gates, audit log |
| **Observability** | 100% trace coverage | OpenTelemetry, structured audit log |

---

## 13. Summary: Tool Integration Quality Gates

Before any tool integration reaches production, it must pass the following **quality gates**:

| Gate | Criterion | Automated Check |
|---|---|---|
| **Schema Validity** | Input/output schemas parse without error; all required fields documented | JSON Schema meta-validation in CI |
| **Description Quality** | Contains [WHAT], [WHEN], [WHEN NOT], [INPUTS], [OUTPUTS], [SIDE EFFECTS] | Regex + LLM-judge scoring > 0.8 |
| **Disambiguation** | Pairwise similarity with sibling tools < 0.7 cosine | Embedding comparison in CI |
| **Latency Compliance** | p99 latency within declared tier | Load test with 100 synthetic queries |
| **Error Handling** | All error classes return typed ToolError; retryable flag accurate | Fault injection testing |
| **Idempotency** | Duplicate calls with same key produce identical state | Replay test suite |
| **Authorization** | Caller scope checked; escalation impossible | Penetration test + formal policy verification |
| **Grounding** | Agent responses using this tool achieve grounding > 0.9 | Eval suite with known-answer queries |
| **Cost Bound** | Per-call cost within declared budget | Cost monitoring in staging |

---

**End of Report.**

This document specifies tools not as informal API wrappers, but as **typed, governed, budget-constrained, observable infrastructure** within a closed-loop agentic control system. Every interface is schema-described, every invocation is traced, every mutation is gated, and every result is verified — the foundational requirements for production-grade agentic AI at enterprise scale.



# Tools as First-Class Typed Infrastructure in Agentic AI Systems

## A Principal-Level Technical Report on Tool Integration, Invocation Protocols, Context Engineering for Tool Affordances, and Production-Grade Orchestration

---

## 1. Executive Summary

Tools constitute the **actuator layer** of an agentic AI system — the boundary at which language model cognition transitions from internal reasoning to **externally observable state mutation and information acquisition**. Without tools, a large language model (LLM) is a closed dynamical system operating over a fixed snapshot of parametric knowledge; with tools, it becomes an **open-loop controller** capable of sensing, acting upon, and modifying real-world state.

This report provides a comprehensive, state-of-the-art (SOTA) treatment of tool integration within agentic architectures, spanning: the formal protocol stack governing tool discovery, invocation, and result ingestion; the context engineering methodology that transforms raw API surfaces into high-fidelity affordance descriptions optimized for model selection accuracy; the evolution from naive prompt-based command generation to typed, schema-governed function calling with structured outputs; the orchestration patterns for single-tool invocation, multi-tool chaining, parallel fan-out, and conditional branching; and the production reliability mechanisms — idempotency, authorization scoping, retry budgets, circuit breaking, observability, and human-in-the-loop gating — that make tool-augmented agents safe for enterprise deployment.

Every concept is formalized with mathematical precision, accompanied by pseudoalgorithmic specification, and justified through explicit trade-off analysis.

---

## 2. Formal Definition: What Constitutes a "Tool" in an Agentic System

### 2.1 Axiomatic Definition

**Definition 2.1 (Tool).** A tool $\tau$ in an agentic system is a **typed, schema-described, externally-executing callable** that bridges the agent's internal reasoning context to an external system, data source, or actuator. Formally:

$$
\tau = \langle \, \text{id}, \, \sigma_{\text{in}}, \, \sigma_{\text{out}}, \, \delta, \, \kappa, \, \pi_{\text{auth}}, \, \mathcal{L}_{\text{latency}}, \, \mathcal{S}_{\text{side-effects}} \, \rangle
$$

where:

| Symbol | Semantics |
|--------|-----------|
| $\text{id}$ | Globally unique, versioned tool identifier (e.g., `flights/search_flights@v2.3.1`) |
| $\sigma_{\text{in}}$ | Input schema — a typed contract (JSON Schema, Protobuf message, or equivalent) specifying required and optional parameters, types, constraints, and validation rules |
| $\sigma_{\text{out}}$ | Output schema — the typed contract of the return payload, including error discriminants |
| $\delta$ | Natural-language **affordance description** — the semantic specification consumed by the LLM to determine applicability; this is the tool's "identity card" within the context window |
| $\kappa$ | Capability classification vector: $\kappa \in \{\texttt{read-only}, \texttt{write}, \texttt{delete}, \texttt{admin}\}$ with associated risk tier |
| $\pi_{\text{auth}}$ | Authorization policy — caller-scoped credentials, OAuth scopes, API key bindings, or human-approval gates |
| $\mathcal{L}_{\text{latency}}$ | Latency class: $\{\texttt{fast} < 100\text{ms}, \, \texttt{medium} < 2\text{s}, \, \texttt{slow} < 30\text{s}, \, \texttt{async}\}$ |
| $\mathcal{S}_{\text{side-effects}}$ | Side-effect declaration: pure query, idempotent mutation, non-idempotent mutation, or irreversible action |

**Remark.** This definition explicitly separates the **model-facing interface** ($\delta$, $\sigma_{\text{in}}$, $\sigma_{\text{out}}$) from the **system-facing infrastructure** ($\pi_{\text{auth}}$, $\mathcal{L}_{\text{latency}}$, $\mathcal{S}_{\text{side-effects}}$). The former is context-engineered for LLM consumption; the latter is mechanically enforced by the runtime.

### 2.2 Tool Taxonomy

Tools are not a monolithic category. A principal-level classification:

| Category | Description | Examples | Risk Profile |
|----------|-------------|----------|-------------|
| **Information Retrieval** | Read-only data acquisition from external sources | `search_flights`, `get_weather`, `query_database` | Low — no state mutation |
| **Computation** | Deterministic transformations the LLM cannot reliably perform internally | `calculate_tax`, `run_regex`, `convert_currency` | Negligible — pure functions |
| **State Mutation** | Write operations that modify external system state | `book_flight`, `send_email`, `create_jira_ticket` | High — requires idempotency and approval gates |
| **Environment Observation** | Introspection of the agent's own execution environment | `read_logs`, `inspect_trace`, `get_test_results` | Medium — information leakage risk |
| **Orchestration** | Meta-tools that spawn, delegate to, or coordinate other agents | `delegate_to_specialist`, `fork_parallel_tasks` | Critical — recursive invocation risk |

---

## 3. The Evolution: From Prompt Hacking to Typed Function Calling

### 3.1 Generation Zero — Prompt-Based Command Synthesis

**Approach.** Early practitioners embedded pseudo-API documentation into the system prompt and instructed the model to emit text resembling a function call. The downstream application layer then parsed this free-text output using regular expressions or fragile string matching.

**Formal Characterization of Failure Modes:**

Let $G(x)$ denote the LLM's generated output given prompt $x$. The prompt-based approach requires:

$$
P\big[\text{parse}(G(x)) \in \mathcal{V}(\sigma_{\text{in}})\big] \geq 1 - \epsilon
$$

where $\mathcal{V}(\sigma_{\text{in}})$ is the set of valid instances of the input schema $\sigma_{\text{in}}$, and $\epsilon$ is the error rate.

**Empirically, $\epsilon$ was unacceptably high** due to:

1. **Syntactic drift**: The model generates near-valid but malformed JSON (trailing commas, unquoted keys, hallucinated fields).
2. **Schema hallucination**: The model invents parameters not present in $\sigma_{\text{in}}$ or omits required ones.
3. **Tool confusion**: When multiple tools share semantic overlap, the model selects incorrectly because disambiguation relied entirely on unstructured prose.
4. **Injection vulnerability**: User input could manipulate the parsing heuristic.

**Verdict:** Generation Zero is a **research curiosity**, not a production pattern. It violates the fundamental requirement that tool invocation must be mechanically verifiable.

### 3.2 Generation One — Native Function Calling (Structured Tool Use)

**Breakthrough.** Model providers (OpenAI, Anthropic, Google, Mistral, Cohere) introduced **native function calling** as a first-class inference mode. The model is trained — via reinforcement learning from human feedback (RLHF) and supervised fine-tuning on tool-use traces — to emit **structured JSON objects** that conform to provider-specified schemas.

**Formal Protocol:**

Given a set of available tools $\mathcal{T} = \{\tau_1, \tau_2, \ldots, \tau_n\}$, each described by its affordance tuple $(\text{id}_i, \sigma_{\text{in},i}, \delta_i)$, the model receives these as part of its context and produces one of two output types:

$$
\text{LLM}(\text{context}, \mathcal{T}) \rightarrow
\begin{cases}
\texttt{text\_response}(m) & \text{if no tool is needed} \\
\texttt{tool\_call}(\text{id}_j, \text{args}_j) & \text{where } \text{args}_j \in \mathcal{V}(\sigma_{\text{in},j})
\end{cases}
$$

**Key Properties of Native Function Calling:**

| Property | Technical Detail |
|----------|-----------------|
| **Structured Output Guarantee** | The model's decoding is constrained (via guided generation, constrained beam search, or grammar-based sampling) to emit valid JSON matching $\sigma_{\text{in}}$ |
| **Parallel Tool Calls** | SOTA models support emitting multiple `tool_call` objects in a single turn, enabling concurrent execution |
| **Tool Choice Control** | The caller can specify `tool_choice: "auto" | "none" | "required" | {specific_tool}` to control model behavior |
| **Schema Validation at Boundary** | The runtime validates `args_j` against $\sigma_{\text{in},j}$ before forwarding to the tool server |

### 3.3 Generation Two — Model Context Protocol (MCP) and Typed Tool Infrastructure

**SOTA Advancement.** MCP (Model Context Protocol) elevates tools from **statically-defined function signatures** embedded in prompts to **dynamically-discoverable, runtime-negotiated capabilities** exposed by external servers.

**Formal Architecture:**

```
┌─────────────┐      JSON-RPC 2.0       ┌─────────────────┐
│  MCP Client  │◄──────────────────────►│   MCP Server     │
│ (Agent Host)  │   (transport-agnostic)  │ (Tool Provider)  │
│               │                         │                  │
│  • discover() │                         │ • tools/list     │
│  • invoke()   │                         │ • tools/call     │
│  • subscribe()│                         │ • resources/list │
│               │                         │ • prompts/list   │
└───────────────┘                         └──────────────────┘
```

**MCP Protocol Operations Relevant to Tools:**

| Operation | Semantics |
|-----------|-----------|
| `tools/list` | Returns paginated catalog of available tools with JSON Schema input definitions, descriptions, and annotations (e.g., `readOnlyHint`, `destructiveHint`, `idempotentHint`, `openWorldHint`) |
| `tools/call` | Invokes a specific tool with validated arguments; returns structured content (text, image, embedded resource) or `isError: true` with diagnostic payload |
| `notifications/tools/list_changed` | Server-initiated notification when tool availability changes — enables **lazy loading** and **dynamic affordance updates** |

**Why MCP is Architecturally Superior to Static Function Definitions:**

$$
\text{MCP Advantage} = \underbrace{\text{Dynamic Discovery}}_{\text{no stale schemas}} + \underbrace{\text{Typed Annotations}}_{\text{risk classification}} + \underbrace{\text{Transport Abstraction}}_{\text{local + remote}} + \underbrace{\text{Change Notifications}}_{\text{live catalog}}
$$

---

## 4. Context Engineering for Tool Affordances

### 4.1 The Affordance Description as a Compiled Artifact

The natural-language description $\delta$ of a tool is **the single most consequential factor** in the model's tool-selection accuracy. This is not a documentation string — it is a **precision-engineered prompt fragment** that must survive compression, compete for attention against all other context elements, and unambiguously convey when the tool should and should not be used.

**Definition 4.1 (Affordance Description Quality Function).** Let $\mathcal{Q}(\delta, q)$ be the probability that the model correctly selects tool $\tau$ when user query $q$ semantically requires $\tau$'s capability. We define affordance quality as:

$$
\mathcal{Q}(\delta) = \mathbb{E}_{q \sim \mathcal{D}_{\text{eval}}} \Big[ \mathbb{1}\big[\text{LLM}(\text{context} \cup \delta, q) = \texttt{tool\_call}(\tau.\text{id}, \cdot)\big] \Big]
$$

where $\mathcal{D}_{\text{eval}}$ is a representative evaluation distribution over queries that should trigger $\tau$.

**Maximizing $\mathcal{Q}(\delta)$ requires the following SOTA principles:**

### 4.2 SOTA Affordance Engineering Principles

#### Principle 1: Semantic Boundary Specification

Every affordance description must specify both **positive** and **negative** selection boundaries:

```
TOOL: search_flights
DESCRIPTION: Search for available commercial airline flights between two
airports on a specific date. Returns flight options with carrier, departure
time, arrival time, duration, number of stops, and price.

USE WHEN: The user requests flight availability, flight options, or airfare
information for a specific route and date.

DO NOT USE WHEN: The user asks about flight status of an already-booked
flight (use get_flight_status instead), or asks about private/charter
aviation (not supported).

INPUTS:
  - origin: IATA airport code (3 letters, e.g., "SFO"). Required.
  - destination: IATA airport code (3 letters, e.g., "NRT"). Required.
  - departure_date: ISO 8601 date (YYYY-MM-DD). Required.
  - cabin_class: Enum ["economy", "business", "first"]. Optional, default "economy".
  - max_stops: Integer [0, 3]. Optional.

RETURNS: Array of FlightOption objects. Empty array if no flights found.
         Never returns error for "no results" — empty array is valid.
```

**Formal Justification.** Negative constraints ("DO NOT USE WHEN") reduce false-positive tool selection by introducing **contrastive signal** that the model can leverage during its internal tool-routing computation. Empirically, this reduces mis-selection rates by 15–30% on multi-tool benchmarks (ToolBench, API-Bank evaluations).

#### Principle 2: Parameter Semantics Over Parameter Names

The model does not "understand" parameters from names alone. Each parameter must carry:

- **Type with constraints** (not just `string` — specify format, regex, enum, range)
- **Semantic grounding** (what the value represents in the domain)
- **Default behavior** (what happens if omitted)
- **Edge case handling** (what constitutes invalid input and how the tool responds)

**Mathematical Formulation — Input Schema Information Density:**

$$
I(\sigma_{\text{in}}) = \sum_{p \in \text{params}} \Big[ H(\text{type}_p) + H(\text{constraint}_p) + H(\text{semantics}_p) + H(\text{default}_p) \Big]
$$

where $H(\cdot)$ denotes the **information content** (in bits) of each specification element. Maximizing $I(\sigma_{\text{in}})$ directly reduces the model's uncertainty during argument generation.

#### Principle 3: Output Schema Specification for Result Interpretation

A tool call is only half the problem. The model must **interpret the result** to continue its reasoning chain. Without output schema specification, the model must infer result structure from examples — a fragile, hallucination-prone process.

**SOTA Practice:** Include a condensed output schema in $\delta$:

```
RETURNS: {
  "flights": [{
    "carrier": "string (airline name)",
    "flight_number": "string",
    "departure": "ISO 8601 datetime",
    "arrival": "ISO 8601 datetime",
    "duration_minutes": "integer",
    "stops": "integer",
    "price_usd": "float"
  }],
  "metadata": {
    "total_results": "integer",
    "search_timestamp": "ISO 8601 datetime"
  }
}
```

#### Principle 4: Token-Budgeted Affordance Compilation

With $n$ tools in the context, the total token cost is:

$$
C_{\text{tools}} = \sum_{i=1}^{n} \text{tokens}(\delta_i) + \text{tokens}(\sigma_{\text{in},i}) + \text{tokens}(\sigma_{\text{out},i})
$$

For a context window of $W$ tokens and a tool budget allocation of $\alpha W$ (typically $\alpha \in [0.05, 0.15]$), the constraint is:

$$
C_{\text{tools}} \leq \alpha W
$$

When $C_{\text{tools}}$ exceeds the budget, **tool pruning and lazy loading** strategies activate (Section 7).

### 4.3 Pseudoalgorithm: Affordance Description Compiler

```
Algorithm 1: COMPILE_TOOL_AFFORDANCE(τ, context_budget)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Input:
  τ          — Tool definition (id, σ_in, σ_out, raw_docs, κ, S_side_effects)
  context_budget — Maximum token allocation for this tool's description

Output:
  δ_compiled — Optimized affordance description

BEGIN
  1. EXTRACT core_capability ← summarize(τ.raw_docs, max_tokens=50)
  2. GENERATE positive_triggers ← enumerate conditions where τ is the correct choice
  3. GENERATE negative_triggers ← enumerate conditions where τ must NOT be selected
     3a. For each sibling tool τ' in same category:
         COMPUTE semantic_overlap(τ, τ')
         IF overlap > θ_confusion THEN
           APPEND disambiguation_clause to negative_triggers
  4. COMPILE parameter_specs:
     FOR each param p in σ_in:
       IF p.required THEN priority ← HIGH
       ELSE priority ← MEDIUM
       EMIT type, constraints, semantic_grounding, default
       PRUNE examples if total_tokens > context_budget * 0.6
  5. COMPILE output_summary:
     IF tokens(σ_out) < context_budget * 0.2 THEN
       EMIT full σ_out
     ELSE
       EMIT compressed schema (top-level keys + types only)
  6. APPEND side_effect_declaration:
     IF S_side_effects ∈ {write, delete, admin} THEN
       EMIT "⚠ THIS TOOL MODIFIES STATE. Confirm intent before invocation."
  7. APPEND latency_hint:
     EMIT "Expected latency: {L_latency}. Plan accordingly."
  8. VALIDATE total_tokens(δ_compiled) ≤ context_budget
     IF exceeded: COMPRESS using extractive summarization on lowest-priority sections
  9. RETURN δ_compiled
END
```

---

## 5. Tool Selection: The Model's Decision Function

### 5.1 Formal Model of Tool Selection

When the agent receives a user query $q$ and has access to tool set $\mathcal{T}$, the tool selection problem is:

$$
\tau^* = \arg\max_{\tau \in \mathcal{T} \cup \{\varnothing\}} \; P(\tau \mid q, \mathcal{C}, \mathcal{T})
$$

where $\mathcal{C}$ is the full context (system prompt, memory, conversation history, retrieved evidence) and $\varnothing$ represents the "no tool needed" option.

**Decomposition of Selection Probability:**

$$
P(\tau \mid q, \mathcal{C}, \mathcal{T}) = \underbrace{P(\text{need\_tool} \mid q, \mathcal{C})}_{\text{Tool Necessity Gate}} \cdot \underbrace{P(\tau \mid q, \mathcal{C}, \mathcal{T}, \text{need\_tool}=\text{true})}_{\text{Tool Discrimination}}
$$

**Tool Necessity Gate** — The model first determines whether any tool is required. Errors here manifest as:
- **False Positive**: The model invokes a tool when it could answer directly from parametric knowledge → unnecessary latency and cost.
- **False Negative**: The model generates a hallucinated answer when it should have consulted a tool → factual incorrectness.

**Tool Discrimination** — Given that a tool is needed, the model must select the correct one. Errors here manifest as:
- **Mis-selection**: Choosing `get_flight_status` when `search_flights` was required.
- **Argument hallucination**: Selecting the correct tool but generating invalid arguments.

### 5.2 SOTA Techniques for Improving Selection Accuracy

#### 5.2.1 Contrastive Tool Descriptions

When multiple tools have overlapping semantic domains, **pairwise contrastive disambiguation** clauses must be injected:

$$
\delta_i^{\text{contrastive}} = \delta_i \cup \Big\{ \text{"Unlike } \tau_j \text{, this tool specifically handles } X \text{ and does NOT handle } Y\text{"} \Big\}
$$

for all $j$ where $\text{sim}(\delta_i, \delta_j) > \theta_{\text{confusion}}$.

#### 5.2.2 Hierarchical Tool Routing

For large tool sets ($|\mathcal{T}| > 20$), presenting all tools simultaneously degrades selection accuracy due to context dilution. The SOTA solution is **two-stage hierarchical routing**:

**Stage 1 — Category Router:**

$$
c^* = \arg\max_{c \in \mathcal{C}_{\text{categories}}} P(c \mid q)
$$

The model selects a tool **category** (e.g., "travel," "finance," "communication") from a compact category catalog.

**Stage 2 — Tool Selector:**

$$
\tau^* = \arg\max_{\tau \in \mathcal{T}_{c^*}} P(\tau \mid q, \mathcal{T}_{c^*})
$$

Only tools within the selected category are loaded into context, dramatically reducing token cost and selection confusion.

```
Algorithm 2: HIERARCHICAL_TOOL_ROUTING(q, T, category_index)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Input:
  q              — User query
  T              — Full tool registry
  category_index — Map from category → [tool_ids]

Output:
  τ* — Selected tool (or ∅)

BEGIN
  1. categories ← list of category names with 1-line descriptions
  2. c* ← LLM_SELECT_CATEGORY(q, categories)   // Low-cost model call
     IF c* = "none" THEN RETURN ∅
  3. T_subset ← LOAD_TOOLS(category_index[c*])  // Lazy load
  4. FOR each τ in T_subset:
       δ_compiled ← COMPILE_TOOL_AFFORDANCE(τ, budget_per_tool)
  5. τ* ← LLM_SELECT_TOOL(q, T_subset)          // Full model call with subset
  6. VALIDATE τ*.args against τ*.σ_in
     IF validation_fails THEN
       REPAIR args via constrained re-generation
  7. RETURN τ*
END
```

#### 5.2.3 Tool Selection via Embedding-Based Pre-Filtering

Before invoking the LLM for tool selection, a **lightweight embedding-based pre-filter** can reduce the candidate set:

$$
\mathcal{T}_{\text{candidates}} = \Big\{ \tau \in \mathcal{T} \;\Big|\; \text{cosine\_sim}\big(\text{embed}(q), \text{embed}(\delta_\tau)\big) > \theta_{\text{relevance}} \Big\}
$$

This is computed at sub-millisecond latency using a vector index over precomputed tool description embeddings, reducing the number of tools injected into the context window.

---

## 6. Tool Chaining and Multi-Step Orchestration

### 6.1 The Composition Problem

Real-world tasks rarely require a single tool invocation. A request such as *"Plan a weekend trip to San Francisco"* requires:

1. `search_flights(origin, destination, date)` → flight options
2. `search_hotels(city, check_in, check_out, budget)` → accommodation options
3. `get_local_events(city, date_range, interests)` → activity recommendations
4. `calculate_budget(flights, hotels, events)` → cost summary
5. `compose_itinerary(flights, hotels, events)` → final deliverable

This is a **directed acyclic graph (DAG) of tool invocations** with data dependencies.

### 6.2 Formal Model: Tool Composition as a Dataflow Graph

**Definition 6.1 (Tool Execution Plan).** A tool execution plan $\mathcal{P}$ is a DAG $G = (V, E)$ where:

- Each vertex $v_i \in V$ represents a tool invocation: $v_i = (\tau_i, \text{args}_i, \text{result}_i)$
- Each directed edge $(v_i, v_j) \in E$ represents a **data dependency**: $\text{result}_i$ feeds into $\text{args}_j$
- Vertices with no incoming edges can execute in **parallel**
- The plan has a **topological ordering** that defines valid execution sequences

**Parallelism Extraction:**

$$
\text{Parallelism}(\mathcal{P}) = \frac{|V|}{\text{critical\_path\_length}(G)}
$$

where the critical path length is the longest chain of sequential dependencies. Maximizing parallelism reduces end-to-end latency.

### 6.3 Plan Generation Strategies

#### Strategy A — Reactive (ReAct-Style) Sequential Planning

The model reasons one step at a time, observing each tool result before deciding the next action.

$$
\text{Loop: } \underbrace{\text{Thought}}_{\text{reason}} \rightarrow \underbrace{\text{Action}}_{\text{tool\_call}} \rightarrow \underbrace{\text{Observation}}_{\text{result}} \rightarrow \text{Thought} \rightarrow \cdots
$$

**Advantages:** Adaptive — can change plan based on intermediate results.
**Disadvantages:** Sequential — no parallelism; high latency; high token cost (full context replayed each iteration).

#### Strategy B — Upfront DAG Planning with Verification

The model generates the full execution plan $\mathcal{P}$ before any tool is called, then a **plan verifier** validates structural correctness, and a **plan executor** runs the DAG with maximal parallelism.

```
Algorithm 3: DAG_PLAN_AND_EXECUTE(q, T, context)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Input:
  q       — User query
  T       — Available tools
  context — Current agent context (memory, session state)

Output:
  result  — Synthesized response incorporating all tool results

BEGIN
  // Phase 1: Plan Generation
  1. P ← LLM_GENERATE_PLAN(q, T, context)
     // P = ordered list of {tool_id, args_template, dependencies}

  // Phase 2: Plan Validation
  2. FOR each step s in P:
       VALIDATE s.tool_id ∈ T
       VALIDATE s.args_template conforms to T[s.tool_id].σ_in
       VALIDATE all s.dependencies reference earlier steps
       VALIDATE no circular dependencies
     IF validation_fails:
       P ← LLM_REPAIR_PLAN(P, validation_errors)

  // Phase 3: Parallel Execution
  3. execution_order ← TOPOLOGICAL_SORT(P)
     results ← {}
     FOR each parallel_batch in execution_order:
       batch_results ← PARALLEL_EXECUTE(parallel_batch, results)
       // Each tool call substitutes dependency results into args_template
       results ← results ∪ batch_results
       
       // Verification gate after each batch
       FOR each r in batch_results:
         IF r.is_error THEN
           recovery_action ← LLM_DIAGNOSE_AND_REPAIR(r, P, results)
           EXECUTE recovery_action  // retry, skip, substitute, or abort

  // Phase 4: Result Synthesis
  4. response ← LLM_SYNTHESIZE(q, results, context)
  5. RETURN response
END
```

#### Strategy C — Hybrid Adaptive Planning (SOTA)

Combines upfront planning with reactive adaptation. The model generates a **provisional plan**, begins execution, and revises the plan when intermediate results invalidate assumptions.

**Formal Specification:**

$$
\mathcal{P}_{t+1} = \text{REPLAN}(\mathcal{P}_t, \text{results}_{1:t}, q, \mathcal{C})
$$

where REPLAN is invoked only when a **deviation detector** identifies that $\text{results}_{1:t}$ invalidate preconditions of remaining steps in $\mathcal{P}_t$.

**Deviation Detection Function:**

$$
\text{deviated}(r_t, \mathcal{P}_t) = \begin{cases}
\texttt{true} & \text{if } r_t.\text{status} = \text{error} \\
\texttt{true} & \text{if } r_t.\text{content} \text{ contradicts preconditions of } \mathcal{P}_t[t+1:] \\
\texttt{false} & \text{otherwise}
\end{cases}
$$

---

## 7. Token-Efficient Tool Context Management

### 7.1 The Tool Context Budget Problem

Let $W$ denote the total context window. The available budget for tool descriptions is:

$$
B_{\text{tools}} = W - B_{\text{system}} - B_{\text{memory}} - B_{\text{history}} - B_{\text{retrieval}} - B_{\text{output\_reserve}}
$$

For typical configurations with $W = 128{,}000$ tokens:

| Component | Typical Allocation |
|-----------|--------------------|
| System prompt + role policy | 2,000–5,000 |
| Memory summaries | 1,000–3,000 |
| Conversation history | 5,000–20,000 |
| Retrieved evidence | 10,000–30,000 |
| **Tool descriptions** | **3,000–15,000** |
| Output reserve | 4,000–8,000 |

With each tool consuming 200–500 tokens (description + schema), the system can simultaneously present **6–75 tools** without exceeding budget.

### 7.2 SOTA Token Optimization Strategies

#### 7.2.1 Lazy Tool Loading via MCP

Tools are **not** pre-loaded into the context. Instead:

1. The agent receives a **compact tool index** (tool names + 10-word summaries, ~20 tokens each).
2. When the model identifies a relevant tool category, the MCP client calls `tools/list` with category filters.
3. Only the selected tool's full affordance description is injected into the context.

**Token savings:**

$$
\text{Savings} = \sum_{i=1}^{n} \text{tokens}(\delta_i^{\text{full}}) - \left[ \sum_{i=1}^{n} \text{tokens}(\delta_i^{\text{summary}}) + \sum_{j \in \text{selected}} \text{tokens}(\delta_j^{\text{full}}) \right]
$$

For $n = 50$ tools with 3 selected per query, this yields ~85% token reduction.

#### 7.2.2 Compressed Schema Representation

Replace verbose JSON Schema with **typed shorthand notation**:

```
// Verbose (127 tokens):
{
  "type": "object",
  "properties": {
    "origin": {"type": "string", "description": "IATA airport code", "pattern": "^[A-Z]{3}$"},
    "destination": {"type": "string", "description": "IATA airport code", "pattern": "^[A-Z]{3}$"},
    "departure_date": {"type": "string", "format": "date", "description": "ISO 8601 date"}
  },
  "required": ["origin", "destination", "departure_date"]
}

// Compressed (41 tokens):
PARAMS:
  origin*: str (IATA 3-letter code, e.g. "SFO")
  destination*: str (IATA 3-letter code, e.g. "NRT")  
  departure_date*: date (YYYY-MM-DD)
  cabin_class: enum[economy|business|first] = economy
  (* = required)
```

**Compression ratio:** $41 / 127 = 0.32$ — a 68% reduction with no information loss for the model.

#### 7.2.3 Result Compression Before Re-Injection

Tool results can be voluminous (e.g., 50 flight options × 200 tokens each = 10,000 tokens). Before injecting results back into context:

```
Algorithm 4: COMPRESS_TOOL_RESULT(result, task_context, budget)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Input:
  result       — Raw tool output
  task_context — Current task objectives and subsequent plan steps
  budget       — Maximum tokens for the compressed result

Output:
  compressed   — Token-budgeted result representation

BEGIN
  1. IF tokens(result) ≤ budget THEN RETURN result  // No compression needed
  
  2. // Structural compression: remove redundant fields
     essential_fields ← IDENTIFY_FIELDS_NEEDED(result.schema, task_context)
     pruned ← PROJECT(result, essential_fields)
  
  3. // Cardinality reduction: top-K most relevant items
     IF result is array AND |result| > K_max THEN
       ranked ← RANK_BY_RELEVANCE(result, task_context)
       pruned ← ranked[:K_max]
       APPEND metadata: "Showing top {K_max} of {|result|} results"
  
  4. // Semantic summarization (last resort)
     IF tokens(pruned) > budget THEN
       compressed ← LLM_SUMMARIZE(pruned, budget,
                      instruction="Preserve all data needed for: {task_context}")
     ELSE
       compressed ← pruned
  
  5. ATTACH provenance: {tool_id, invocation_time, result_count, compression_applied}
  6. RETURN compressed
END
```

---

## 8. Production Reliability for Tool Execution

### 8.1 The Reliability Stack

Tool invocations cross the boundary between the agent's reasoning process and external systems. This boundary is where **reliability engineering becomes non-negotiable**.

```
┌──────────────────────────────────────────────────────────────────┐
│                    TOOL RELIABILITY STACK                        │
├──────────────────────────────────────────────────────────────────┤
│  Layer 7: Human-in-the-Loop Gates                               │
│    ├─ Approval required for κ ∈ {write, delete, admin}          │
│    └─ Configurable per-tool, per-user, per-risk-tier            │
├──────────────────────────────────────────────────────────────────┤
│  Layer 6: Authorization & Credential Scoping                    │
│    ├─ Caller-scoped tokens (not agent-owned)                    │
│    ├─ Least-privilege OAuth scopes per tool                     │
│    └─ Credential rotation and audit logging                     │
├──────────────────────────────────────────────────────────────────┤
│  Layer 5: Idempotency & Mutation Safety                         │
│    ├─ Idempotency keys for all write operations                 │
│    ├─ Pre-invocation state snapshot for rollback                │
│    └─ Compensating action registry                              │
├──────────────────────────────────────────────────────────────────┤
│  Layer 4: Retry Policy & Circuit Breaking                       │
│    ├─ Exponential backoff with jitter                           │
│    ├─ Per-tool retry budget: max_retries, max_retry_duration    │
│    └─ Circuit breaker: open after N consecutive failures        │
├──────────────────────────────────────────────────────────────────┤
│  Layer 3: Timeout & Deadline Propagation                        │
│    ├─ Per-tool timeout from L_latency class                     │
│    ├─ Deadline inheritance from parent agent loop               │
│    └─ Graceful cancellation on deadline expiry                  │
├──────────────────────────────────────────────────────────────────┤
│  Layer 2: Input Validation & Schema Enforcement                 │
│    ├─ JSON Schema validation against σ_in                       │
│    ├─ Argument sanitization (injection prevention)              │
│    └─ Required field check, type coercion, range validation     │
├──────────────────────────────────────────────────────────────────┤
│  Layer 1: Observability & Tracing                               │
│    ├─ Structured trace: tool_id, args_hash, latency, status     │
│    ├─ Cost attribution per invocation                           │
│    └─ Anomaly detection: unusual argument patterns, latency     │
└──────────────────────────────────────────────────────────────────┘
```

### 8.2 Idempotency Enforcement

**Definition 8.1 (Idempotent Tool Invocation).** A tool invocation $f(\text{args}, k)$ with idempotency key $k$ is idempotent if:

$$
f(\text{args}, k) = f(\text{args}, k) \circ f(\text{args}, k) \quad \forall \text{args}, k
$$

That is, repeated invocations with the same key produce the same result and side effects as a single invocation.

**Implementation Strategy:**

```
Algorithm 5: IDEMPOTENT_TOOL_INVOKE(τ, args, agent_trace_id)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Input:
  τ              — Tool to invoke
  args           — Validated arguments
  agent_trace_id — Unique identifier for this agent execution step

Output:
  result — Tool result (possibly cached from prior identical invocation)

BEGIN
  1. idempotency_key ← SHA256(τ.id || canonical_serialize(args) || agent_trace_id)
  
  2. cached_result ← IDEMPOTENCY_STORE.GET(idempotency_key)
     IF cached_result ≠ NULL THEN
       LOG("Idempotent cache hit", tool=τ.id, key=idempotency_key)
       RETURN cached_result
  
  3. IF τ.S_side_effects ∈ {write, delete} THEN
       // Acquire distributed lock to prevent concurrent duplicate mutations
       lock ← ACQUIRE_LOCK(idempotency_key, ttl=τ.L_latency * 3)
       IF lock_failed THEN
         RETURN AWAIT_EXISTING_INVOCATION(idempotency_key)
  
  4. result ← EXECUTE_WITH_TIMEOUT(τ.endpoint, args, timeout=τ.L_latency)
  
  5. IDEMPOTENCY_STORE.SET(idempotency_key, result, ttl=IDEMPOTENCY_RETENTION)
  
  6. IF lock THEN RELEASE_LOCK(lock)
  
  7. RETURN result
END
```

### 8.3 Retry Policy with Exponential Backoff and Jitter

**Mathematical Specification:**

For retry attempt $r$ (where $r \in [0, R_{\max}]$):

$$
\text{delay}(r) = \min\Big(\text{base} \cdot 2^r + \text{Uniform}(0, \text{jitter\_max}), \; \text{delay\_cap}\Big)
$$

Typical configuration per latency class:

| Latency Class | $R_{\max}$ | base (ms) | jitter_max (ms) | delay_cap (ms) |
|---------------|------------|-----------|-----------------|----------------|
| fast | 3 | 50 | 100 | 500 |
| medium | 3 | 200 | 500 | 5,000 |
| slow | 2 | 1,000 | 2,000 | 30,000 |
| async | 5 | 5,000 | 10,000 | 120,000 |

**Circuit Breaker State Machine:**

$$
\text{CLOSED} \xrightarrow{N \text{ consecutive failures}} \text{OPEN} \xrightarrow{\text{cooldown expiry}} \text{HALF-OPEN} \xrightarrow{\text{probe success}} \text{CLOSED}
$$
$$
\text{HALF-OPEN} \xrightarrow{\text{probe failure}} \text{OPEN}
$$

### 8.4 Human-in-the-Loop Gating for State-Mutating Tools

**Policy Function:**

$$
\text{gate}(\tau, \text{args}, \text{user\_context}) = \begin{cases}
\texttt{auto-approve} & \text{if } \tau.\kappa = \texttt{read-only} \\
\texttt{auto-approve} & \text{if } \tau.\kappa = \texttt{write} \wedge \text{risk\_score}(\text{args}) < \theta_{\text{low}} \\
\texttt{require-confirmation} & \text{if } \tau.\kappa = \texttt{write} \wedge \text{risk\_score}(\text{args}) \geq \theta_{\text{low}} \\
\texttt{require-approval} & \text{if } \tau.\kappa \in \{\texttt{delete}, \texttt{admin}\} \\
\end{cases}
$$

where $\text{risk\_score}$ factors in: monetary amount, irreversibility, number of affected entities, and deviation from user's historical patterns.

```
Algorithm 6: HUMAN_GATED_TOOL_EXECUTION(τ, args, user_session)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

BEGIN
  1. gate_decision ← EVALUATE_GATE_POLICY(τ, args, user_session)
  
  2. CASE gate_decision OF
       auto-approve:
         PROCEED to execution
       
       require-confirmation:
         PRESENT to user: "I will {action_description}. Proceed? [Yes/No/Modify]"
         user_response ← AWAIT_USER_INPUT(timeout=300s)
         IF user_response = "No" THEN ABORT with user_cancellation
         IF user_response = "Modify" THEN
           args ← COLLECT_MODIFIED_ARGS(user_session)
           RESTART from step 1
         // else proceed
       
       require-approval:
         approval_request ← CREATE_APPROVAL_REQUEST(τ, args, user_session)
         ENQUEUE approval_request to APPROVAL_QUEUE
         approval ← AWAIT_APPROVAL(timeout=approval_sla)
         IF approval.denied THEN ABORT with denial_reason
         IF approval.timeout THEN ABORT with escalation
  
  3. result ← IDEMPOTENT_TOOL_INVOKE(τ, args, trace_id)
  4. LOG_AUDIT_TRAIL(τ, args, result, gate_decision, approver)
  5. RETURN result
END
```

---

## 9. Tool Error Handling and Failure Recovery

### 9.1 Error Taxonomy

Tool failures are not uniform. A production-grade agent must handle each class differently:

| Error Class | Example | Recovery Strategy |
|------------|---------|-------------------|
| **Transient** | HTTP 429 (rate limited), 503 (service unavailable) | Retry with backoff |
| **Input Error** | 400 (bad request), schema validation failure | Re-generate arguments via LLM, possibly with error feedback |
| **Authorization** | 401/403 | Escalate to user for re-authentication; do not retry |
| **Not Found** | 404, empty result set | Reformulate query or inform user of absence |
| **Timeout** | Deadline exceeded | Retry if within budget, else degrade gracefully |
| **Semantic Error** | Tool returns data but it doesn't match expectations | LLM interprets and adapts plan |
| **Catastrophic** | Tool server completely unreachable | Circuit break, fall back to alternative tool or cached data |

### 9.2 Self-Healing Argument Repair

When a tool returns a validation error, the agent can **repair** its arguments by feeding the error back into the LLM:

$$
\text{args}_{\text{repaired}} = \text{LLM}\big(q, \tau.\delta, \tau.\sigma_{\text{in}}, \text{args}_{\text{original}}, \text{error\_message}\big)
$$

```
Algorithm 7: SELF_HEALING_TOOL_CALL(τ, args, q, max_repair_attempts=2)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

BEGIN
  FOR attempt = 1 TO max_repair_attempts + 1:
    result ← INVOKE(τ, args)
    IF result.success THEN RETURN result
    
    IF result.error_class = TRANSIENT THEN
      WAIT(backoff_delay(attempt))
      CONTINUE  // Retry same args
    
    IF result.error_class = INPUT_ERROR THEN
      args ← LLM_REPAIR_ARGUMENTS(
        original_args=args,
        error=result.error_message,
        schema=τ.σ_in,
        query=q,
        instruction="Fix the arguments based on the error. Do not guess — if unsure, ask the user."
      )
      CONTINUE  // Retry with repaired args
    
    IF result.error_class ∈ {AUTH, CATASTROPHIC} THEN
      BREAK  // Non-recoverable
  
  // All attempts exhausted
  RETURN GRACEFUL_DEGRADATION(τ, q, last_error=result.error)
END
```

---

## 10. Observability and Cost Attribution

### 10.1 Structured Trace Schema for Tool Invocations

Every tool invocation generates a structured trace record:

```protobuf
message ToolInvocationTrace {
  string trace_id = 1;
  string parent_agent_trace_id = 2;
  string tool_id = 3;
  string tool_version = 4;
  google.protobuf.Struct args = 5;         // Sanitized (no PII)
  google.protobuf.Struct result_summary = 6;
  
  enum Status { SUCCESS = 0; RETRIED = 1; REPAIRED = 2; FAILED = 3; CIRCUIT_BROKEN = 4; }
  Status status = 7;
  
  int32 attempt_count = 8;
  int64 latency_ms = 9;
  int32 input_tokens_consumed = 10;        // Tokens for affordance description
  int32 output_tokens_from_result = 11;    // Tokens for injected result
  float estimated_cost_usd = 12;
  
  string gate_decision = 13;              // auto-approve, require-confirmation, etc.
  string approver_id = 14;               // If human approval was required
  
  google.protobuf.Timestamp invoked_at = 15;
  google.protobuf.Timestamp completed_at = 16;
}
```

### 10.2 Cost Model

Total cost of a tool invocation includes both **direct** and **indirect** costs:

$$
\text{Cost}_{\text{total}}(\tau) = \underbrace{C_{\text{API}}(\tau)}_{\text{External API cost}} + \underbrace{C_{\text{tokens}}(\delta_\tau)}_{\text{Context tokens for description}} + \underbrace{C_{\text{tokens}}(r_\tau)}_{\text{Context tokens for result}} + \underbrace{C_{\text{compute}}(\text{retries})}_{\text{Retry overhead}}
$$

where $C_{\text{tokens}}(x) = \text{tokens}(x) \cdot \text{price\_per\_token}$ at the model's pricing tier.

**Optimization Objective:**

$$
\min_{\mathcal{T}_{\text{loaded}}} \sum_{\tau \in \mathcal{T}_{\text{loaded}}} C_{\text{tokens}}(\delta_\tau) \quad \text{subject to} \quad P(\text{correct\_selection} \mid \mathcal{T}_{\text{loaded}}) \geq 1 - \epsilon
$$

This is the **tool budget optimization problem**: load the minimum set of tool descriptions that maintains selection accuracy above threshold $1 - \epsilon$.

---

## 11. Multi-Tool Protocol Integration Architecture

### 11.1 The Complete Protocol Stack

```
                    ┌─────────────────────────────────────┐
                    │         USER / APPLICATION           │
                    │    (Web UI, CLI, API Consumer)       │
                    └───────────────┬─────────────────────┘
                                    │ JSON-RPC 2.0
                                    │ (request/response, notifications)
                    ┌───────────────▼─────────────────────┐
                    │        AGENT ORCHESTRATOR            │
                    │  ┌─────────────────────────────┐    │
                    │  │    Prefill Compiler          │    │
                    │  │    Plan Generator            │    │
                    │  │    Tool Router               │    │
                    │  │    Result Synthesizer        │    │
                    │  │    Memory Manager            │    │
                    │  └─────────────────────────────┘    │
                    └──┬──────────┬──────────┬────────────┘
                       │          │          │
              gRPC/    │    MCP   │    MCP   │   gRPC/
              Protobuf │  (stdio) │  (SSE)  │   Protobuf
                       │          │          │
                ┌──────▼──┐ ┌────▼────┐ ┌──▼──────────┐
                │ Internal │ │ Local   │ │ Remote      │
                │ Services │ │ Tool    │ │ Tool        │
                │ (search, │ │ Servers │ │ Servers     │
                │  memory, │ │ (file   │ │ (APIs,      │
                │  eval)   │ │  system,│ │  SaaS,      │
                │          │ │  git,   │ │  databases) │
                │          │ │  shell) │ │             │
                └──────────┘ └────────┘ └─────────────┘
```

### 11.2 Protocol Selection Rationale

| Boundary | Protocol | Justification |
|----------|----------|---------------|
| User ↔ Orchestrator | JSON-RPC 2.0 | Human-readable, widely supported, supports batch and notifications, natural fit for request-response agent interactions |
| Orchestrator ↔ Internal Services | gRPC/Protobuf | Binary encoding for minimal latency, strong typing via `.proto` contracts, bi-directional streaming for long-running operations, native deadline propagation |
| Orchestrator ↔ Tool Servers | MCP (JSON-RPC over stdio/SSE) | Standardized discovery (`tools/list`, `resources/list`), change notifications, transport-agnostic, ecosystem interoperability with growing MCP server catalog |

---

## 12. End-to-End Worked Example: Travel Agent Tool Chain

### 12.1 Scenario

**User Query:** *"Find me a flight from San Francisco to Tokyo next Tuesday, and a good hotel in Shinjuku for 3 nights under $200/night."*

### 12.2 Execution Trace

```
Step 1: QUERY DECOMPOSITION
  Subquery A: "Flight SFO → NRT/HND, departure 2025-07-22"
  Subquery B: "Hotel in Shinjuku, check-in 2025-07-22, check-out 2025-07-25, max $200/night"
  Dependency: None (A and B are independent → parallel execution)

Step 2: TOOL ROUTING
  Subquery A → ROUTE to tool category "travel/flights"
    → Lazy load: search_flights affordance (287 tokens)
    → Selected: search_flights
    → Args: {origin: "SFO", destination: "NRT", departure_date: "2025-07-22"}
    → Validation: ✓ (all required fields present, IATA codes valid)
  
  Subquery B → ROUTE to tool category "travel/accommodation"
    → Lazy load: search_hotels affordance (312 tokens)
    → Selected: search_hotels
    → Args: {city: "Tokyo", neighborhood: "Shinjuku",
              check_in: "2025-07-22", check_out: "2025-07-25",
              max_price_per_night_usd: 200}
    → Validation: ✓

Step 3: PARALLEL EXECUTION
  [Thread A] IDEMPOTENT_INVOKE(search_flights, args_A, trace_001)
    → Gate: auto-approve (read-only)
    → Latency: 847ms (medium class, within budget)
    → Result: 23 flight options
    → COMPRESS_RESULT: top 5 by price, projected to {carrier, flight_number,
       departure, arrival, stops, price_usd} → 412 tokens
  
  [Thread B] IDEMPOTENT_INVOKE(search_hotels, args_B, trace_002)
    → Gate: auto-approve (read-only)
    → Latency: 1,203ms (medium class, within budget)
    → Result: 15 hotel options
    → COMPRESS_RESULT: top 5 by rating within budget, projected to {name,
       rating, price_per_night, distance_to_station} → 389 tokens

Step 4: RESULT SYNTHESIS
  Context injected: compressed results (801 tokens total)
  LLM synthesizes natural-language response with structured recommendations
  Response includes: 3 best flight options, 3 best hotel options, total cost estimates

Step 5: TRACE EMISSION
  Traces: 2 ToolInvocationTrace records
  Total tool context cost: 287 + 312 + 412 + 389 = 1,400 tokens
  Total latency: max(847, 1203) = 1,203ms (parallel execution)
  External API cost: $0.002 (flight API) + $0.001 (hotel API) = $0.003
```

---

## 13. Evaluation Infrastructure for Tool Use Quality

### 13.1 Tool Use Quality Metrics

| Metric | Definition | Target |
|--------|-----------|--------|
| **Tool Selection Precision** | $\frac{\text{correct selections}}{\text{total selections}}$ | $\geq 0.95$ |
| **Tool Selection Recall** | $\frac{\text{correct selections}}{\text{queries requiring tools}}$ | $\geq 0.92$ |
| **Argument Validity Rate** | $\frac{\text{schema-valid args on first attempt}}{\text{total tool calls}}$ | $\geq 0.90$ |
| **Self-Repair Success Rate** | $\frac{\text{successful repairs}}{\text{repair attempts}}$ | $\geq 0.80$ |
| **End-to-End Task Completion** | $\frac{\text{fully completed multi-tool tasks}}{\text{total multi-tool tasks}}$ | $\geq 0.85$ |
| **Tool Hallucination Rate** | $\frac{\text{calls to non-existent tools or fabricated results}}{\text{total calls}}$ | $\leq 0.01$ |
| **Unnecessary Tool Call Rate** | $\frac{\text{tool calls when direct answer was possible}}{\text{total calls}}$ | $\leq 0.05$ |

### 13.2 Continuous Evaluation Pipeline

```
Algorithm 8: TOOL_USE_EVALUATION_CI(eval_suite, model_config, tool_registry)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Input:
  eval_suite    — Set of (query, expected_tool, expected_args, expected_outcome) tuples
  model_config  — Model version, temperature, tool_choice settings
  tool_registry — Current tool definitions with affordance descriptions

Output:
  quality_report — Pass/fail with per-metric scores

BEGIN
  FOR each (q, τ_expected, args_expected, outcome_expected) in eval_suite:
    1. result ← RUN_AGENT(q, model_config, tool_registry)
    2. EVALUATE:
       selection_correct ← (result.tool_selected == τ_expected)
       args_valid ← VALIDATE(result.args, τ_expected.σ_in)
       args_semantically_correct ← SEMANTIC_MATCH(result.args, args_expected)
       outcome_correct ← SEMANTIC_MATCH(result.final_response, outcome_expected)
    3. RECORD metrics
  
  quality_report ← AGGREGATE_METRICS(all_results)
  
  FOR each metric in quality_report:
    IF metric.value < metric.threshold THEN
      quality_report.status ← FAIL
      ALERT("Tool use quality regression detected", metric)
  
  RETURN quality_report
END
```

**Integration Point:** This pipeline executes on every change to:
- Tool affordance descriptions ($\delta$)
- Tool schema definitions ($\sigma_{\text{in}}$, $\sigma_{\text{out}}$)
- Model version or configuration
- System prompt or role policy

---

## 14. Security and Authorization Architecture

### 14.1 Principle of Least Privilege for Tool Access

**Axiom:** An agent must never hold broader credentials than the minimum required for the current user's authorized actions.

**Implementation:**

$$
\text{effective\_permissions}(\text{agent}, \tau) = \text{user\_permissions}(\tau) \cap \text{tool\_max\_permissions}(\tau) \cap \text{session\_scope}
$$

```
┌────────────────────────────────────────────────────┐
│              AUTHORIZATION FLOW                     │
│                                                    │
│  User authenticates → Session token issued         │
│       │                                            │
│       ▼                                            │
│  Agent receives user-scoped credential proxy       │
│  (NOT the raw credential)                          │
│       │                                            │
│       ▼                                            │
│  Tool invocation includes:                         │
│    - Credential proxy (exchanged at tool boundary) │
│    - Agent identity (for audit)                    │
│    - Invocation context (for rate limiting)        │
│       │                                            │
│       ▼                                            │
│  Tool server validates:                            │
│    - Credential proxy → user identity              │
│    - User authorized for this operation?           │
│    - Rate limit not exceeded?                      │
│    - Invocation context consistent?                │
│       │                                            │
│       ▼                                            │
│  Execute or reject with typed error                │
└────────────────────────────────────────────────────┘
```

### 14.2 Argument Injection Prevention

The agent's generated arguments must be **sanitized** before tool execution to prevent injection attacks:

$$
\text{args}_{\text{safe}} = \text{SANITIZE}(\text{args}_{\text{raw}}, \tau.\sigma_{\text{in}})
$$

where SANITIZE enforces:
- **Type coercion**: Strings that should be integers are cast or rejected.
- **Length bounds**: No argument exceeds the maximum length specified in the schema.
- **Pattern matching**: Regex-constrained fields (e.g., IATA codes) are validated.
- **Escape sequences**: SQL, shell, and HTML injection patterns are neutralized at the tool gateway, not at the model layer.

---

## 15. Comparative Analysis: Static Function Calling vs. MCP-Based Tool Infrastructure

| Dimension | Static Function Calling | MCP-Based Tool Infrastructure |
|-----------|------------------------|-------------------------------|
| **Discovery** | Compile-time; tools hardcoded in prompt | Runtime; `tools/list` with pagination and filtering |
| **Schema Evolution** | Requires prompt update and redeployment | Server-side schema update + `list_changed` notification |
| **Multi-Server** | Manual aggregation of tool definitions | Standardized client connects to N servers simultaneously |
| **Resource Access** | Tools only | Tools + Resources + Prompt Templates |
| **Annotations** | Ad-hoc via description text | Structured: `readOnlyHint`, `destructiveHint`, `idempotentHint`, `openWorldHint` |
| **Transport** | HTTP/REST (typically) | stdio (local), SSE (remote), extensible |
| **Interoperability** | Provider-specific formats | Open standard; growing ecosystem |
| **Token Efficiency** | All tools loaded upfront | Lazy loading via discovery → selective injection |
| **Scalability** | Degrades beyond ~20 tools | Hierarchical routing + lazy loading scales to hundreds |

**Verdict:** MCP is the **architecturally superior** approach for any system with more than 5 tools or any requirement for runtime tool catalog evolution.

---

## 16. Summary of Key Architectural Invariants

| Invariant | Enforcement Mechanism |
|-----------|----------------------|
| Every tool has a typed input and output schema | JSON Schema validation at invocation gateway |
| No tool modifies state without explicit side-effect declaration | MCP annotations + gateway policy enforcement |
| Tool descriptions are compiled artifacts, not raw documentation | Affordance compiler (Algorithm 1) with token budgets |
| Tool results are provenance-tagged and compressed before context injection | Result compression pipeline (Algorithm 4) |
| All state-mutating invocations are idempotent | Idempotency key computation and cache (Algorithm 5) |
| Human approval gates are mechanically enforced for high-risk operations | Gate policy function with configurable thresholds |
| Tool use quality is continuously measured | CI evaluation pipeline (Algorithm 8) with regression alerts |
| Retry budgets are bounded and jittered | Per-tool retry policy with circuit breaker state machine |
| Agent credentials never exceed user permissions | Caller-scoped credential proxy architecture |
| Tool context cost is bounded by explicit token budget | Lazy loading, compression, hierarchical routing |

---

## 17. Conclusion

Tools transform an LLM from a **stateless reasoning engine** into an **embodied agent** capable of sensing and actuating within real-world systems. However, the difference between a demo-quality tool integration and a production-grade tool infrastructure is vast. The former concatenates API documentation into a prompt and hopes for the best; the latter engineers a **typed protocol stack** with formal discovery, schema-governed invocation, idempotent mutation, bounded retry, human-in-the-loop safety gates, token-budgeted context management, and continuous quality evaluation.

The key insight is that **tool integration is not a prompting problem — it is a systems engineering problem** that happens to have an LLM as the decision-making component. The LLM selects and parameterizes tools; everything else — validation, execution, reliability, security, observability, cost control — is mechanical infrastructure that must be designed, tested, and operated with the same rigor as any production distributed system.

The evolution from prompt-based command synthesis (Generation Zero) through native function calling (Generation One) to MCP-based discoverable tool infrastructure (Generation Two) represents a maturation trajectory from **brittle string manipulation** to **typed, versioned, interoperable protocol-level integration** — the only foundation capable of supporting enterprise-scale agentic systems.

---

*End of Technical Report*